# ML-05 — Feature Leakage Check

This notebook demonstrates that label-derived columns (`trend_direction`, `trend_pct`) leak the target when included as features, and verifies that the actual model features are safe.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Prepare the label
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Fill numerics
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Rows: {len(df):,}  |  Declining: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.1%})")

## 2. The leakage demonstration

*Train a model WITH trend_pct as a feature. If it achieves near-perfect AUC, that's leakage proof.*

In [ ]:
# --- LEAKY MODEL: includes trend_pct ---
leaky_features = ["impressions_90d", "clicks_90d", "sessions_90d",
                   "content_age_days", "avg_position", "ctr",
                   "trend_pct"]  # <-- THIS IS THE LEAK

X_leaky = df[leaky_features].fillna(0)
y = df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

leaky_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
leaky_model.fit(X_train, y_train)
leaky_proba = leaky_model.predict_proba(X_test)[:, 1]
leaky_auc = roc_auc_score(y_test, leaky_proba)

print(f"🚨 LEAKY MODEL (with trend_pct):  AUC = {leaky_auc:.4f}")
print(f"   This is {'near-perfect — LEAKAGE CONFIRMED' if leaky_auc > 0.95 else 'suspiciously high'}!")
print(f"   trend_pct directly encodes whether the page declined.")

## 3. The clean model (no leakage)

*Train without trend_pct or trend_direction. AUC should be moderate, not perfect.*

In [ ]:
# --- CLEAN MODEL: no label-derived features ---
clean_features = ["impressions_90d", "clicks_90d", "sessions_90d",
                   "content_age_days", "avg_position", "ctr",
                   "days_with_impressions", "engagement_rate", "scroll_rate"]

X_clean = df[clean_features].fillna(0)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

clean_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clean_model.fit(X_train_c, y_train_c)
clean_proba = clean_model.predict_proba(X_test_c)[:, 1]
clean_auc = roc_auc_score(y_test_c, clean_proba)

print(f"✅ CLEAN MODEL (without trend_pct):  AUC = {clean_auc:.4f}")
print(f"   This is a moderate, honest AUC — the model learns real patterns,")
print(f"   not the label itself.")

## 4. Side-by-side comparison

In [ ]:
print("\n" + "="*55)
print(f"{'Model':<30s} {'AUC':>10s}   Verdict")
print("="*55)
print(f"{'Leaky (with trend_pct)':<30s} {leaky_auc:>10.4f}   🚨 LEAKAGE")
print(f"{'Clean (without trend_pct)':<30s} {clean_auc:>10.4f}   ✅ Honest")
print("="*55)
print(f"\nGap: {leaky_auc - clean_auc:.4f} AUC points")
print(f"The leaky model's near-perfect AUC proves that trend_pct")
print(f"directly encodes the label. It must NEVER be a feature.")

## 5. Verify the actual model feature list is clean

*Check that the production feature lists in ml_utils.py exclude all leaky columns.*

In [ ]:
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

all_model_features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
leaky_columns = {"trend_direction", "trend_pct", "is_declining_label"}

found_leaks = leaky_columns.intersection(set(all_model_features))

print(f"Model feature count: {len(all_model_features)}")
print(f"Leaky columns in feature list: {found_leaks if found_leaks else 'NONE ✅'}")
print(f"\n{'✅ PASS' if not found_leaks else '❌ FAIL'}: "
      f"The production feature list is {'clean' if not found_leaks else 'LEAKY — FIX IMMEDIATELY'}.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.